# Configuration 1 -> 
## Sentence transformer embedding model


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/sentence_transformers/all_chunks.csv")

In [5]:
df.head()

,id,text
0,/Users/midhunln/Documents/rag20march_with_eval...,Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...
1,/Users/midhunln/Documents/rag20march_with_eval...,Securities and Exchange Board of India (Issue ...
2,/Users/midhunln/Documents/rag20march_with_eval...,Page 2 of 132 \n \n4. With the issuance of thi...
3,/Users/midhunln/Documents/rag20march_with_eval...,"suffered thereunder, any right, privilege, obl..."
4,/Users/midhunln/Documents/rag20march_with_eval...,regulations and bidding portal. \n7. All liste...


In [6]:
df["id"][0]

'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_0_0'

In [7]:
df.info

<bound method DataFrame.info of                                                      id  \
0     /Users/midhunln/Documents/rag20march_with_eval...   
1     /Users/midhunln/Documents/rag20march_with_eval...   
2     /Users/midhunln/Documents/rag20march_with_eval...   
3     /Users/midhunln/Documents/rag20march_with_eval...   
4     /Users/midhunln/Documents/rag20march_with_eval...   
...                                                 ...   
1103  /Users/midhunln/Documents/rag20march_with_eval...   
1104  /Users/midhunln/Documents/rag20march_with_eval...   
1105  /Users/midhunln/Documents/rag20march_with_eval...   
1106  /Users/midhunln/Documents/rag20march_with_eval...   
1107  /Users/midhunln/Documents/rag20march_with_eval...   

                                                   text  
0     Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...  
1     Securities and Exchange Board of India (Issue ...  
2     Page 2 of 132 \n \n4. With the issuance of thi...  
3     suffered thereunder, 

In [8]:
df_sampled = df.sample(frac = 0.1)

In [9]:
df_sampled.head()

,id,text
35,/Users/midhunln/Documents/rag20march_with_eval...,Page 15 of 132 \n \nChapter 5: Compensation to...
268,/Users/midhunln/Documents/rag20march_with_eval...,Page 117 of 132
643,/Users/midhunln/Documents/rag20march_with_eval...,"income recognition, asset classification, prov..."
377,/Users/midhunln/Documents/rag20march_with_eval...,2026 – 27 Top 1000 listed entities \n \n3. ESG...
478,/Users/midhunln/Documents/rag20march_with_eval...,requirements which is not specified by SEBI un...


In [10]:
!pip install langchain-ollama


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model = "llama3")

/Users/midhunln/Documents/rag20march_with_eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llama3")

def llm_call(text):
    prompt = f"""
You are a query generator. Based on the text below, generate a concise search query
that a user would likely use to find this information. The output must only have the query and nothing else.
It is very important that the output is only the query and nothing else.

Text:
{text}
"""
    response = chat.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


# Apply row-wise correctly
df_sampled["query"] = df_sampled["text"].apply(llm_call)

In [13]:
df_sampled["query"].iloc[2]

'"chartered accountants report"'

In [14]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/sentence_transformers/chunk_to_query_mapping.csv", index = False)

In [15]:
df_sampled.head()

,id,text,query
35,/Users/midhunln/Documents/rag20march_with_eval...,Page 15 of 132 \n \nChapter 5: Compensation to...,"""IPO allotment failure compensation"""
268,/Users/midhunln/Documents/rag20march_with_eval...,Page 117 of 132,"""page 117"""
643,/Users/midhunln/Documents/rag20march_with_eval...,"income recognition, asset classification, prov...","""chartered accountants report"""
377,/Users/midhunln/Documents/rag20march_with_eval...,2026 – 27 Top 1000 listed entities \n \n3. ESG...,"""ESG disclosures value chain"""
478,/Users/midhunln/Documents/rag20march_with_eval...,requirements which is not specified by SEBI un...,"""SEBI circular requirements"""


# Construct the qrels dataset for rankx

In [16]:
qrels = {}

for _, row in df_sampled.iterrows():
    query = row["query"]
    doc_id = row["id"]

    if query not in qrels:
        qrels[query] = {}

    qrels[query][doc_id] = 1  # relevance score

In [17]:
qrels

{'"IPO allotment failure compensation"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_14_35': 1},
 '"page 117"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_116_18': 1},
 '"chartered accountants report"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_113_43': 1},
 '"ESG disclosures value chain"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulati

# Start retriever


In [18]:
!pip install langchain-pinecone


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
import sys
sys.path.append('/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval')

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

from Repositories.pinecone_repository import PineconeRepository

In [20]:
from Repositories.pinecone_repository import PineconeConfig
from src.sentence_transformer_embedding import SentenceTransformerEmbedding
from src.sparse_embedding import SentenceTransformerSparseEmbedding

In [21]:
config = PineconeConfig()

dense_embedding_strategy = SentenceTransformerEmbedding(config)

sparse_embedding_strategy = SentenceTransformerSparseEmbedding(config)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6448.34it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 40419.39it/s]
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` 

In [22]:
import os

In [23]:
pinecone_repository = PineconeRepository(
            api_key=os.getenv("PINECONE_API_KEY"),
            environment=os.getenv("PINECONE_ENVIRONMENT", "us-east-1-aws"),
            dense_embedding_strategy=dense_embedding_strategy,
            sparse_embedding_strategy=sparse_embedding_strategy,
            pinecone_config=config
        )

In [24]:
df_sampled["relevant_docs"] = df_sampled["query"].apply(pinecone_repository.query_vector_store_for_rankx)

In [25]:
df_sampled.to_csv("/Users/midhunln/Documents/rag20march_with_eval/notebooks/datasets/sentence_transformers/chunk_query_relevant_docs.csv", index = False)

In [26]:
df_sampled.head()

,id,text,query,relevant_docs
35,/Users/midhunln/Documents/rag20march_with_eval...,Page 15 of 132 \n \nChapter 5: Compensation to...,"""IPO allotment failure compensation""",[{'id': '/Users/midhunln/Documents/rag20march_...
268,/Users/midhunln/Documents/rag20march_with_eval...,Page 117 of 132,"""page 117""",[{'id': '/Users/midhunln/Documents/rag20march_...
643,/Users/midhunln/Documents/rag20march_with_eval...,"income recognition, asset classification, prov...","""chartered accountants report""",[{'id': '/Users/midhunln/Documents/rag20march_...
377,/Users/midhunln/Documents/rag20march_with_eval...,2026 – 27 Top 1000 listed entities \n \n3. ESG...,"""ESG disclosures value chain""",[{'id': '/Users/midhunln/Documents/rag20march_...
478,/Users/midhunln/Documents/rag20march_with_eval...,requirements which is not specified by SEBI un...,"""SEBI circular requirements""",[{'id': '/Users/midhunln/Documents/rag20march_...


In [27]:
df_sampled["relevant_docs"].iloc[0]

[{'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_14_35',
  'score': 21.7768345},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_32_44',
  'score': 19.8449631},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_14_36',
  'score': 16.3281155},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_52_42',
  'score': 16.1422958},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_15_38',
  'score': 15.6715603},
 {'id': '/Users

In [28]:
qrels_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    true_doc = row["id"]   # single doc id string
    qrels_dict[qid] = {true_doc: 1}

In [29]:
run_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    
    run_dict[qid] = {
        doc["id"]: float(doc["score"])
        for doc in row["relevant_docs"]
    }

In [30]:
from ranx import Qrels, Run, evaluate

qrels = Qrels(qrels_dict)
run = Run(run_dict)

metrics = [
    "mrr",
    "ndcg@10",
    "recall@5",
    "recall@10",
    "precision@5"
]

results = evaluate(qrels, run, metrics)
print(results)

{'mrr': np.float64(0.6078435578435578), 'ndcg@10': np.float64(0.66452595030322), 'recall@5': np.float64(0.8108108108108109), 'recall@10': np.float64(0.8378378378378378), 'precision@5': np.float64(0.16216216216216214)}


In [31]:
final_result = pd.Series(results)


In [32]:
final_result

mrr            0.607844
ndcg@10        0.664526
recall@5       0.810811
recall@10      0.837838
precision@5    0.162162
dtype: float64